# 05 --- Validation & Review

Validators, review rules, human corrections, and the approval workflow. Runs on a real PDF using Gemini Flash 2.5.

## Why post-extraction quality gates matter

LLMs are remarkably good at reading documents, but they are not perfect. Common failure modes include:

- **Hallucination** — the model invents a value that doesn't exist in the document (e.g., fabricating an invoice number).
- **Misinterpretation** — the model reads a value but gets the format wrong (e.g., confusing European and US decimal formats, or reading "Net 30" as a number).
- **Omission** — the model skips a field that is clearly present in the document.
- **Low confidence** — the model returns a value but isn't sure about it (especially with poor-quality scans or ambiguous text).

DocuFlow addresses this with two layers of quality gates that run automatically after extraction:

### Validators vs Review Rules

**Validators** check structural correctness — they look at the extraction result and flag specific problems:
- Are all required fields present and non-null?
- Do fields that should have evidence actually have it?
- Are there type mismatches or empty values?

Validators annotate each field with a `validation_status` and accumulate errors, but they never stop the pipeline. The philosophy is: give the downstream consumer all the information, let them decide what to do.

**Review rules** are decision gates — they examine the result and decide whether a human needs to look at it. If any rule fires, `needs_review=True` and the reason is recorded. Rules check things like overall confidence thresholds, per-field confidence, missing critical fields, or even call another LLM to verify business logic.

### The human-in-the-loop pattern

When a result is flagged for review, DocuFlow supports a full correction and approval workflow:
1. A reviewer inspects the flagged fields.
2. They can **correct** individual values — the original is preserved, the correction is logged with who/when/why.
3. They **approve** or **reject** the result — this is final and timestamped.
4. The full chain (extraction → evidence → validation → review → correction → approval) is captured in the **provenance** record — a complete audit trail per field.

In [ ]:
import os

os.environ["GEMINI_API_KEY"] = os.environ.get("GEMINI_API_KEY", "")

PDF_PATH = "data/sample_invoice.pdf"

from pydantic import BaseModel, Field
from typing import Optional, List


class LineItem(BaseModel):
    description: str = Field(description="Line item description")
    quantity: Optional[float] = Field(default=None, description="Quantity ordered")
    unit_price: Optional[float] = Field(default=None, description="Price per unit")
    tax_rate: Optional[float] = Field(default=None, description="Tax rate percentage for this item")
    amount: float = Field(description="Line item total amount")


class Invoice(BaseModel):
    supplier_name: str = Field(description="Name of the supplier or vendor")
    invoice_number: str = Field(description="Invoice reference number")
    invoice_date: str = Field(description="Date the invoice was issued (YYYY-MM-DD)")
    due_date: Optional[str] = Field(default=None, description="Payment due date (YYYY-MM-DD)")
    po_number: Optional[str] = Field(default=None, description="Purchase order number")
    currency: str = Field(default="USD", description="Currency code (ISO 4217)")
    bill_to_company: Optional[str] = Field(default=None, description="Company billed")
    ship_to_company: Optional[str] = Field(default=None, description="Company shipped to")
    subtotal: Optional[float] = Field(default=None, description="Amount before tax")
    tax_rate: Optional[float] = Field(default=None, description="Tax rate as percentage")
    tax_amount: Optional[float] = Field(default=None, description="Tax amount")
    total: float = Field(description="Total amount including tax")
    payment_terms: Optional[str] = Field(default=None, description="Payment terms")
    line_items: Optional[List[LineItem]] = Field(default=None, description="Individual line items")


print(f"Schema: {len(Invoice.model_fields)} fields")
print(f"PDF: {PDF_PATH}")

## Extract first

Run a basic extraction to get a result we can validate and review.

In [ ]:
from docuflow import DocumentPipeline

pipeline = DocumentPipeline(parser="pdfplumber", model="gemini/gemini-2.5-flash")
result = pipeline.run_sync(PDF_PATH, schema=Invoice)

print(f"Extracted {len(result.fields)} fields, confidence: {result.confidence:.2f}")

## Validators

Validators run after extraction and annotate fields with a validation status. They accumulate errors -- they do not raise exceptions.

In [ ]:
from docuflow.validation import RequiredFields, EvidenceRequired, TypeValidation

validators = [
    RequiredFields(["supplier_name", "invoice_number", "total"]),
    EvidenceRequired(["total", "supplier_name"]),
    TypeValidation(),
]

for v in validators:
    errors = v.validate(result)
    name = v.__class__.__name__
    if errors:
        for e in errors:
            print(f"  [{e.severity.upper():7}] {name} -> {e.field_name}: {e.message}")
    else:
        print(f"  [OK     ] {name}: all checks passed")

## Review rules

Review rules decide whether a result needs human review. If any rule fires, `needs_review=True` and a reason is recorded.

In [ ]:
from docuflow.review import (
    OverallConfidenceBelow,
    FieldConfidenceBelow,
    AnyFieldConfidenceBelow,
    FieldMissing,
    NoEvidence,
)

rules = [
    OverallConfidenceBelow(0.80),
    FieldConfidenceBelow({"total": 0.90, "supplier_name": 0.85}),
    AnyFieldConfidenceBelow(0.85),
    FieldMissing(["total", "invoice_number"]),
    NoEvidence(["total", "supplier_name"]),
]

for rule in rules:
    reason = rule.check(result)
    name = rule.__class__.__name__
    status = "FLAGGED" if reason else "OK"
    print(f"  [{status:7}] {name}")
    if reason:
        print(f"            {reason}")

## Pipeline with validators + review rules

Validators and review rules can be passed directly to `DocumentPipeline` so they run automatically after extraction.

In [ ]:
full_pipeline = DocumentPipeline(
    parser="pdfplumber",
    model="gemini/gemini-2.5-flash",
    validators=[
        RequiredFields(["supplier_name", "total"]),
        EvidenceRequired(["total"]),
    ],
    review_rules=[
        OverallConfidenceBelow(0.7),
        AnyFieldConfidenceBelow(0.6),
    ],
)

result2 = full_pipeline.run_sync(PDF_PATH, schema=Invoice)

print(f"needs_review: {result2.needs_review}")
print(f"review_reasons: {result2.review_reasons}")

## Human corrections

A reviewer can correct individual fields. The original value is preserved, and every correction is logged.

In [ ]:
import copy

review_result = copy.deepcopy(result)

print(f"Before: total = {review_result.fields['total'].value}")
print(f"Corrected: {review_result.fields['total'].corrected}")

review_result.correct_field(
    "total",
    new_value=review_result.fields["total"].value,
    corrected_by="analyst",
    reason="Verified against source document",
)

print(f"\nAfter correction:")
print(f"  value: {review_result.fields['total'].value}")
print(f"  original_value: {review_result.fields['total'].original_value}")
print(f"  corrected: {review_result.fields['total'].corrected}")

print(f"\nCorrections log:")
for c in review_result.corrections:
    print(f"  {c.field_name}: {c.old_value} -> {c.new_value} by {c.corrected_by}")

## Approve / Reject

Once reviewed, a result is approved or rejected. Approving a second time raises a `ValueError`.

In [ ]:
review_result.approve(approved_by="analyst")

print(f"review_status: {review_result.review_status}")
print(f"reviewed_by: {review_result.reviewed_by}")
print(f"reviewed_at: {review_result.reviewed_at}")

In [ ]:
try:
    review_result.approve(approved_by="someone_else")
except ValueError as e:
    print(f"Expected error: {e}")

## Provenance

Full audit trail per field: extraction source, validation status, review outcome, and correction history.

In [ ]:
prov = review_result.provenance()

for fname, p in prov.items():
    print(f"  {fname}: value={p.value}, confidence={p.confidence:.2f}, "
          f"corrected={p.corrected}, review={p.review_status}")

In [ ]:
p = review_result.provenance("total")["total"]

print(f"Full provenance for 'total':")
print(f"  value: {p.value}")
print(f"  source_text: {p.source_text}")
print(f"  page: {p.page}")
print(f"  model: {p.model_name}, parser: {p.parser_name}")
print(f"  validation: {p.validation_status}")
print(f"  review: {p.review_status}, by: {p.reviewed_by}")
print(f"  corrected: {p.corrected}, by: {p.corrected_by}")